In [1]:
import re
import pandas as pd

from dandi.dandiapi import DandiAPIClient

DANDISET_ID = "000409"
VERSION = "0.260309.1324"

records = []

with DandiAPIClient() as client:
    dandiset = client.get_dandiset(
        dandiset_id=DANDISET_ID,
        version_id=VERSION,
    )

    for asset in dandiset.get_assets():
        path = asset.path

        # 只保留已处理的行为 + 电生理 NWB
        if not path.endswith("_desc-processed_behavior+ecephys.nwb"):
            continue

        match = re.search(
            r"sub-(?P<subject>[^/]+).*?_ses-(?P<session>[^_]+)_desc-",
            path,
        )

        if match is None:
            print("无法解析：", path)
            continue

        records.append(
            {
                "subject": match.group("subject"),
                "session": match.group("session"),
                "asset_path": path,
            }
        )

session_inventory = (
    pd.DataFrame(records)
    .drop_duplicates()
    .sort_values(["subject", "session"])
    .reset_index(drop=True)
)

print("Processed NWB files:", len(session_inventory))
print("Subjects:", session_inventory["subject"].nunique())
print("Sessions:", session_inventory["session"].nunique())

display(session_inventory.head(10))

Processed NWB files: 459
Subjects: 139
Sessions: 459


,subject,session,asset_path
0,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...
1,CSH-ZAD-011,03cf52f6-fba6-4743-a42e-dd1ac3072343,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-03cf52f6-f...
2,CSH-ZAD-011,5b44c40f-80f4-44fb-abfb-c7f19e27a6ca,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-5b44c40f-8...
3,CSH-ZAD-011,da188f2c-553c-4e04-879b-c9ea2d1b9a93,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-da188f2c-5...
4,CSH-ZAD-019,49e0ab27-827a-4c91-bcaa-97eea27a1b8d,sub-CSH-ZAD-019/sub-CSH-ZAD-019_ses-49e0ab27-8...
5,CSH-ZAD-019,5adab0b7-dfd0-467d-b09d-43cb7ca5d59c,sub-CSH-ZAD-019/sub-CSH-ZAD-019_ses-5adab0b7-d...
6,CSH-ZAD-019,7f6b86f9-879a-4ea2-8531-294a221af5d0,sub-CSH-ZAD-019/sub-CSH-ZAD-019_ses-7f6b86f9-8...
7,CSH-ZAD-019,edd22318-216c-44ff-bc24-49ce8be78374,sub-CSH-ZAD-019/sub-CSH-ZAD-019_ses-edd22318-2...
8,CSH-ZAD-022,1eac875c-feaa-4a30-b148-059b954b11d8,sub-CSH-ZAD-022/sub-CSH-ZAD-022_ses-1eac875c-f...
9,CSH-ZAD-022,41431f53-69fd-4e3b-80ce-ea62e03bf9c7,sub-CSH-ZAD-022/sub-CSH-ZAD-022_ses-41431f53-6...


In [2]:
print("DataFrame形状：", session_inventory.shape)
print("Processed NWB数量：", len(session_inventory))
print("Subject数量：", session_inventory["subject"].nunique())
print("Session数量：", session_inventory["session"].nunique())

display(session_inventory.tail(10))

DataFrame形状： (459, 3)
Processed NWB数量： 459
Subject数量： 139
Session数量： 459


,subject,session,asset_path
449,ibl-witten-29,0cc486c3-8c7b-494d-aa04-b70e2690bcba,sub-ibl-witten-29/sub-ibl-witten-29_ses-0cc486...
450,ibl-witten-29,58b1e920-cfc8-467e-b28b-7654a55d0977,sub-ibl-witten-29/sub-ibl-witten-29_ses-58b1e9...
451,ibl-witten-29,69a0e953-a643-4f0e-bb26-dc65af3ea7d7,sub-ibl-witten-29/sub-ibl-witten-29_ses-69a0e9...
452,ibl-witten-29,72028382-a869-4745-bacf-cb8789e16953,sub-ibl-witten-29/sub-ibl-witten-29_ses-720283...
453,ibl-witten-29,821f1883-27f3-411d-afd3-fb8241bbc39a,sub-ibl-witten-29/sub-ibl-witten-29_ses-821f18...
454,ibl-witten-29,86b6ba67-c1db-4333-add0-f8105ea6e363,sub-ibl-witten-29/sub-ibl-witten-29_ses-86b6ba...
455,ibl-witten-29,af55d16f-0e31-4073-bdb5-26da54914aa2,sub-ibl-witten-29/sub-ibl-witten-29_ses-af55d1...
456,ibl-witten-29,d2832a38-27f6-452d-91d6-af72d794136c,sub-ibl-witten-29/sub-ibl-witten-29_ses-d2832a...
457,ibl-witten-29,e9fc0a2d-c69d-44d1-9fa3-314782387cae,sub-ibl-witten-29/sub-ibl-witten-29_ses-e9fc0a...
458,ibl-witten-32,08102cfc-a040-4bcf-b63c-faa0f4914a6f,sub-ibl-witten-32/sub-ibl-witten-32_ses-08102c...


In [3]:
session_inventory.to_csv(
    "../results/session_inventory.csv",
    index=False
)

In [1]:
from pathlib import Path
import pandas as pd

# 当前 Notebook 位于项目的 notebooks 文件夹
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

RESULTS_DIR = PROJECT_ROOT / "results"
CACHE_DIR = PROJECT_ROOT / "cache" / "remfile"
DATA_DIR = PROJECT_ROOT / "data"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

session_inventory = pd.read_csv(
    RESULTS_DIR / "session_inventory.csv"
)

print("Project root:", PROJECT_ROOT)
print("Cache directory:", CACHE_DIR)
print("Inventory shape:", session_inventory.shape)

display(session_inventory.head())

Project root: D:\Ephys_analysis\ibl-population-dynamics
Cache directory: D:\Ephys_analysis\ibl-population-dynamics\cache\remfile
Inventory shape: (459, 3)


,subject,session,asset_path
0,CSH-ZAD-001,3e7ae7c0-fe8b-487c-9354-036236fa1010,sub-CSH-ZAD-001/sub-CSH-ZAD-001_ses-3e7ae7c0-f...
1,CSH-ZAD-011,03cf52f6-fba6-4743-a42e-dd1ac3072343,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-03cf52f6-f...
2,CSH-ZAD-011,5b44c40f-80f4-44fb-abfb-c7f19e27a6ca,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-5b44c40f-8...
3,CSH-ZAD-011,da188f2c-553c-4e04-879b-c9ea2d1b9a93,sub-CSH-ZAD-011/sub-CSH-ZAD-011_ses-da188f2c-5...
4,CSH-ZAD-019,49e0ab27-827a-4c91-bcaa-97eea27a1b8d,sub-CSH-ZAD-019/sub-CSH-ZAD-019_ses-49e0ab27-8...


In [26]:
import h5py
import remfile

from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO

DANDISET_ID = "000409"
VERSION = "0.260309.1324"

# 示例 session
preferred_session = "034e726f-b35f-41e0-8d6c-a22cc32391fb"

matched = session_inventory[
    session_inventory["session"] == preferred_session
]

if len(matched) > 0:
    test_row = matched.iloc[0]
else:
    test_row = session_inventory.iloc[0]

test_subject = test_row["subject"]
test_session = test_row["session"]
test_asset_path = test_row["asset_path"]

print("Subject:", test_subject)
print("Session:", test_session)
print("Asset:", test_asset_path)

Subject: CSHL045
Session: 034e726f-b35f-41e0-8d6c-a22cc32391fb
Asset: sub-CSHL045/sub-CSHL045_ses-034e726f-b35f-41e0-8d6c-a22cc32391fb_desc-processed_behavior+ecephys.nwb


In [27]:
with DandiAPIClient() as client:
    dandiset = client.get_dandiset(
        dandiset_id=DANDISET_ID,
        version_id=VERSION,
    )

    asset = dandiset.get_asset_by_path(
        path=test_asset_path
    )

    s3_url = asset.get_content_url(
        follow_redirects=1,
        strip_query=True,
    )

print("Remote URL obtained.")

Remote URL obtained.


In [28]:
disk_cache = remfile.DiskCache(str(CACHE_DIR))

rem_file = remfile.File(
    s3_url,
    disk_cache=disk_cache,
)

h5py_file = h5py.File(
    rem_file,
    mode="r",
)

io = NWBHDF5IO(
    file=h5py_file,
    load_namespaces=True,
)

nwbfile = io.read()

print("NWB opened successfully.")
print("Session ID:", nwbfile.session_id)

NWB opened successfully.
Session ID: 034e726f-b35f-41e0-8d6c-a22cc32391fb


In [29]:
print(nwbfile.session_id)

034e726f-b35f-41e0-8d6c-a22cc32391fb


In [30]:
print("Units:", nwbfile.units is not None)
print("Trials:", nwbfile.trials is not None)

if nwbfile.units is not None:
    print("\nNumber of units:", len(nwbfile.units))
    print("\nUnits columns:")
    for column in nwbfile.units.colnames:
        print("-", column)

if nwbfile.trials is not None:
    print("\nNumber of trials:", len(nwbfile.trials))
    print("\nTrials columns:")
    for column in nwbfile.trials.colnames:
        print("-", column)

Units: True
Trials: True

Number of units: 1346

Units columns:
- unit_name
- spike_distances_from_probe_tip_um
- sliding_rp_violation
- missed_spikes_estimate
- max_spike_amplitude_uV
- probe_name
- waveform_mean
- median_spike_amplitude_uV
- electrodes
- spike_amplitude_std_dB
- spike_count
- peak_to_trough_duration_ms
- presence_ratio
- firing_rate
- spike_amplitudes_uV
- ibl_quality_score
- noise_cutoff
- kilosort2_label
- distance_from_probe_tip_um
- cumulative_drift_um_per_hour
- min_spike_amplitude_uV
- isi_violations_ratio
- max_electrode
- spike_times
- presence_ratio_std
- cluster_uuid
- rp_violation

Number of trials: 589

Trials columns:
- start_time
- stop_time
- quiescence_period
- gabor_stimulus_onset_time
- auditory_cue_time
- wheel_movement_onset_time
- choice_registration_time
- feedback_time
- gabor_stimulus_offset_time
- gabor_stimulus_contrast
- gabor_stimulus_side
- mouse_wheel_choice
- is_mouse_rewarded
- reward_volume_uL
- probability_left
- block_type
- block_i

In [31]:
print("Number of electrodes:", len(nwbfile.electrodes))

print("\nElectrodes columns:")
for column in nwbfile.electrodes.colnames:
    print(
        "-",
        column,
        ":",
        nwbfile.electrodes[column].description
    )

Number of electrodes: 768

Electrodes columns:
- location : Location of the electrode (channel).
- group : Reference to the ElectrodeGroup.
- electrode_name : Unique identifier derived from probe contact ids.
- contact_shapes : Contact shape per electrode as defined by the probe.
- channel_name : The name of this channel, showing all recording streams for the electrode (e.g., 'AP379,LF379').
- rel_x : Relative x coordinate on the probe (µm).
- rel_y : Relative y coordinate on the probe (µm).
- probe_name : IBL probe name in canonical format (e.g., 'Probe00', 'Probe01').
- contact_ids : Channel IDs as specified by the probe manufacturer.
- shank_ids : Shank ID for each electrode (for multi-shank probes).
- adc_group : ADC group index from SpikeGLX metadata.
- adc_sample_order : ADC sample order from SpikeGLX metadata.
- group_name : Name of the ElectrodeGroup.
- x : x coordinate of the channel location in the brain.
- y : y coordinate of the channel location in the brain.
- z : z coordi

In [32]:
electrodes_df = nwbfile.electrodes.to_dataframe()

print("Electrodes table shape:", electrodes_df.shape)
display(electrodes_df.head())

Electrodes table shape: (768, 18)


,location,group,electrode_name,contact_shapes,channel_name,rel_x,rel_y,probe_name,contact_ids,shank_ids,adc_group,adc_sample_order,group_name,x,y,z,imp,filtering
id,,,,,,,,,,,,,,,,,,
0,Medial amygdalar nucleus,Probe01 pynwb.ecephys.ElectrodeGroup at 0x2376...,e0,square,"AP0,LF0",16.0,0.0,Probe01,e0,,0,0,Probe01,6550.0,6626.0,3876.0,NaN,
1,Medial amygdalar nucleus,Probe01 pynwb.ecephys.ElectrodeGroup at 0x2376...,e1,square,"AP1,LF1",48.0,0.0,Probe01,e1,,1,0,Probe01,6550.0,6626.0,3876.0,NaN,
2,Medial amygdalar nucleus,Probe01 pynwb.ecephys.ElectrodeGroup at 0x2376...,e2,square,"AP2,LF2",0.0,20.0,Probe01,e2,,0,1,Probe01,6550.0,6605.0,3876.0,NaN,
3,Medial amygdalar nucleus,Probe01 pynwb.ecephys.ElectrodeGroup at 0x2376...,e3,square,"AP3,LF3",32.0,20.0,Probe01,e3,,1,1,Probe01,6550.0,6605.0,3876.0,NaN,
4,Medial amygdalar nucleus,Probe01 pynwb.ecephys.ElectrodeGroup at 0x2376...,e4,square,"AP4,LF4",16.0,40.0,Probe01,e4,,0,2,Probe01,6550.0,6583.0,3876.0,NaN,


In [33]:
first_unit_electrodes = nwbfile.units["electrodes"][0]

print(type(first_unit_electrodes))
display(first_unit_electrodes)

<class 'pandas.DataFrame'>


,location,group,electrode_name,contact_shapes,channel_name,rel_x,rel_y,probe_name,contact_ids,shank_ids,adc_group,adc_sample_order,group_name,x,y,z,imp,filtering
id,,,,,,,,,,,,,,,,,,
384,Zona incerta,Probe00 pynwb.ecephys.ElectrodeGroup at 0x2376...,e0,square,"AP0,LF0",16.0,0.0,Probe00,e0,,0,0,Probe00,7700.0,4412.0,3489.0,NaN,
385,Zona incerta,Probe00 pynwb.ecephys.ElectrodeGroup at 0x2376...,e1,square,"AP1,LF1",48.0,0.0,Probe00,e1,,1,0,Probe00,7700.0,4412.0,3489.0,NaN,
386,Zona incerta,Probe00 pynwb.ecephys.ElectrodeGroup at 0x2376...,e2,square,"AP2,LF2",0.0,20.0,Probe00,e2,,0,1,Probe00,7700.0,4391.0,3482.0,NaN,
387,Zona incerta,Probe00 pynwb.ecephys.ElectrodeGroup at 0x2376...,e3,square,"AP3,LF3",32.0,20.0,Probe00,e3,,1,1,Probe00,7700.0,4391.0,3482.0,NaN,
388,Zona incerta,Probe00 pynwb.ecephys.ElectrodeGroup at 0x2376...,e4,square,"AP4,LF4",16.0,40.0,Probe00,e4,,0,2,Probe00,7700.0,4370.0,3475.0,NaN,
389,Zona incerta,Probe00 pynwb.ecephys.ElectrodeGroup at 0x2376...,e5,square,"AP5,LF5",48.0,40.0,Probe00,e5,,1,2,Probe00,7700.0,4370.0,3475.0,NaN,
390,Zona incerta,Probe00 pynwb.ecephys.ElectrodeGroup at 0x2376...,e6,square,"AP6,LF6",0.0,60.0,Probe00,e6,,0,3,Probe00,7700.0,4350.0,3468.0,NaN,
391,Zona incerta,Probe00 pynwb.ecephys.ElectrodeGroup at 0x2376...,e7,square,"AP7,LF7",32.0,60.0,Probe00,e7,,1,3,Probe00,7700.0,4350.0,3468.0,NaN,
392,Zona incerta,Probe00 pynwb.ecephys.ElectrodeGroup at 0x2376...,e8,square,"AP8,LF8",16.0,80.0,Probe00,e8,,0,4,Probe00,7700.0,4329.0,3461.0,NaN,


In [34]:
io.close()
h5py_file.close()
rem_file.close()

print("Remote NWB closed.")

Remote NWB closed.
